# 22 — Sentence Transformers
**Goal:** Generate dense sentence embeddings using transformer models.

A sentence embedding is one dense vector (384 or 768 dims) that represents the *meaning* of a whole sentence. Sentence Transformers fine-tune a BERT-style model with a pooling layer on top, so "Python is great for NLP" and "I love Python" land close together even though they share almost no words. This is the step up from Ch. 21: GloVe gives one vector per *word*; here we get one vector per *sentence*.

**Why it matters for resumes / ATS:** the core ATS operation is matching a resume line against a job-description line, and those rarely share exact vocabulary ("ML" vs "machine learning", "pytorch" vs "torch"). Sentence-level embeddings make that comparison semantic instead of lexical — and the resume-vs-JD cells in this chapter are the template the rest of the project builds on.

## 1. What is a Sentence Embedding?

A sentence embedding compresses a variable-length sentence into a **fixed-size vector**: the transformer encodes each token, then a pooling layer (mean or CLS) collapses them into one vector. Fixed size matters — downstream code (similarity, clustering, storage) never needs to know how long the input was.

**What the code does:** prints a cheat-sheet contrasting sentence embeddings with BoW/TF-IDF:
- **semantics, not keyword overlap** — paraphrases get high similarity;
- **fixed-size vector** regardless of sentence length;
- dimension is model-specific: `all-MiniLM-L6-v2` → 384, larger models → 768+.

**Why it matters for resumes / ATS:** a skills line "Python, TensorFlow, SQL" and a JD line "experience with Python and deep learning frameworks" produce close vectors even with one word in common — BoW cosine would score them near zero.

In [ ]:
print('''A sentence embedding is a dense vector representing an entire sentence's meaning:
"Python is great for NLP"  →  [0.23, -0.45, 0.12, ...]  (384 or 768 dims)

Unlike BoW/TF-IDF:
- Captures semantics, not just keyword overlap
- "I love Python" ≈ "Python is my favorite" (high similarity)
- Fixed-size vector regardless of sentence length''')

## 2. Loading Sentence Transformers

`SentenceTransformer("all-MiniLM-L6-v2")` loads a pre-trained model (first call downloads ~90 MB) and wraps encoding, pooling, and normalization behind one object. MiniLM-L6 is this project's workhorse: 6 transformer layers, 384-dim output, fast on CPU.

**What the code does:** loads the model, reports its configuration, then encodes one resume phrase:
- `model.max_seq_length` — expected **256** (the model's documented truncation limit);
- `model.get_sentence_embedding_dimension()` — expected **384**;
- `model.encode("Python developer with NLP experience")` — a single vector of shape `(384,)`, and the cell prints its first 5 values (model-specific floats, roughly in `[-1, 1]`).

**Try it:** swap in `"all-mpnet-base-v2"` (768 dims, better quality, ~2× slower) or `"paraphrase-MiniLM-L6-v2"` (tuned for paraphrase similarity) and compare the reported numbers.

In [ ]:
from sentence_transformers import SentenceTransformer, util
# all-MiniLM-L6-v2: fast, good quality, 384 dims
print("Loading all-MiniLM-L6-v2...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model loaded. Max sequence length: {model.max_seq_length}")
print(f"Output dimension: {model.get_sentence_embedding_dimension()}")

# Quick test
emb = model.encode("Python developer with NLP experience")
print(f"Embedding shape: {emb.shape}")
print(f"First 5 values: {emb[:5].round(3)}")

## 3. Semantic Similarity

Semantic similarity = **cosine similarity** between two sentence embeddings. `util.cos_sim(A, B)` returns a matrix of pairwise scores; it is the one primitive the whole matching stack reduces to.

**What the code does:** encodes four resume-like sentences, computes the full 4×4 matrix via `util.cos_sim(embeddings, embeddings)`, and prints each pair once with a ✓ marker when similarity exceeds **0.5**. Expected ranking:
- S1 (Python + ML) vs S2 (data scientist, Python + NLP) — **highest**: both are Python/ML profiles;
- S1 vs S3 (Java backend, Spring Boot) — **lowest**: disjoint skills;
- S3 vs S4 (ML engineer, deep learning) — in between: shared "engineer/ML" flavor, different tools.

The 0.5 threshold is a guess, not a law — production pipelines tune it on labeled data, and Ch. 23 gives the tooling for exactly that.

In [ ]:
sentences = [
    "Python developer with machine learning experience",
    "Data scientist skilled in Python and NLP",
    "Java backend engineer with Spring Boot",
    "ML engineer building deep learning models",
]
embeddings = model.encode(sentences)
similarities = util.cos_sim(embeddings, embeddings)

print("Similarity matrix:")
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        sim = similarities[i][j].item()
        marker = "✓" if sim > 0.5 else " "
        print(f"  {marker} S{i+1} vs S{j+1}: {sim:.3f}  ({sentences[i][:30]}... / {sentences[j][:30]}...)")

## 4. Resume vs JD Matching with Sentence Transformers

This is the production use case in miniature: one resume embedding, several job embeddings, pick the highest cosine. No keyword lists, no hand-written rules — the model does the semantic work.

**What the code does:** encodes a data-science resume and three jobs, then `util.cos_sim(res_emb, job_embs)[0]` extracts the row of scores. It prints each job with `✓✓` when the score exceeds 0.5. Expected outcome:
- Job 1 (Data Scientist — Python, ML, NLP, TensorFlow) — **highest**: overlaps the resume on nearly every term;
- Job 3 (ML Engineer — Deep Learning, PyTorch, MLOps) — second: same neighborhood, different tooling;
- Job 2 (Java Backend — Spring, Microservices, AWS) — **lowest**: unrelated stack.

**Try it:** add a job that mentions the same tools in a different domain ("Python automation tester") and watch the score drop — the model separates "Python as ML" from "Python as glue code".

In [ ]:
resume = "Senior data scientist with 5 years Python, TensorFlow, NLP experience"
jobs = [
    "Data Scientist — Python, ML, NLP, TensorFlow required",
    "Java Backend Developer — Spring, Microservices, AWS",
    "ML Engineer — Deep Learning, PyTorch, MLOps",
]
res_emb = model.encode(resume)
job_embs = model.encode(jobs)
scores = util.cos_sim(res_emb, job_embs)[0]

print("Resume vs Job matching:")
for i, job in enumerate(jobs):
    match = "✓✓" if scores[i] > 0.5 else "  "
    print(f"  {match} Job {i+1}: {scores[i]:.3f} — {job}")

## 5. Finding the Best Matching Resume Section

Ranking whole resumes is coarse — recruiters also want to know *which part* of a resume matched. The fix: embed each section separately and compare it to the JD.

**What the code does:** stores four sections (`summary`, `experience`, `skills`, `education`) in a dict, encodes the JD once, then encodes each section and prints its cosine vs the JD. Expected ordering:
- `skills` ("Python, TensorFlow, PyTorch, SQL, AWS") and `summary` ("data scientist with Python and ML") — **top**: direct hits on the JD's "Python ML engineer with NLP and cloud";
- `experience` — middle: mentions NLP but also off-topic detail;
- `education` ("M.S. Computer Science, Stanford") — **lowest**: no skill terms at all.

Section-level matching is how a real ATS answers "why did this resume match?" — each score becomes an explainable highlight.

In [ ]:
resume_sections = {
    "summary": "Data scientist with Python and ML expertise",
    "experience": "Built NLP pipelines at Google, reduced latency by 40%",
    "skills": "Python, TensorFlow, PyTorch, SQL, AWS",
    "education": "M.S. Computer Science, Stanford University",
}
jd = "Looking for Python ML engineer with NLP and cloud experience"

jd_emb = model.encode(jd)
print("Section relevance to JD:")
for name, text in resume_sections.items():
    sec_emb = model.encode(text)
    sim = util.cos_sim(jd_emb, sec_emb).item()
    print(f"  {name:12s} {sim:.3f} — {text[:40]}...")

## Key Insight: Sentence Transformers are the gold standard for semantic matching. The rest of this project uses them extensively.

**One model, one primitive — cosine over sentence embeddings — replaces keyword matching, synonym lists, and hand-tuned rules.**

The chapter's three patterns (pairwise similarity, resume-vs-JD, section-vs-JD) cover most of what a semantic ATS needs, and the 0.5 thresholds are deliberately arbitrary. Ch. 23 turns those choices into evidence: it benchmarks Sentence Transformers against Ch. 19/21's static vectors on resume-specific pairs, so the model pick and threshold become measured decisions instead of guesses.